In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import ace_tools as tools

# File paths
file_paths = {
    "sports_management_agencies": "C:/Users/nadza/Downloads/sports_management_agencies.csv",
    "sponsors_dataset": "C:/Users/nadza/Downloads/sponsors_dataset_.csv",
    "communication_boxes": "C:/Users/nadza/Downloads/communication_boxes_africa_updated.csv",
}

# Load datasets
datasets = {name: pd.read_csv(path) for name, path in file_paths.items()}

# Display basic info for each dataset
dataset_info = {}
for name, df in datasets.items():
    dataset_info[name] = {
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values (%)": (df.isnull().sum().sum() / df.size) * 100,
        "Duplicated Rows": df.duplicated().sum(),
        "Data Types": df.dtypes.to_dict(),
    }

# Display dataset information
dataset_info_df = pd.DataFrame(dataset_info).T
tools.display_dataframe_to_user(name="Dataset Overview", dataframe=dataset_info_df)


ModuleNotFoundError: No module named 'ace_tools'

In [ ]:
# Remove duplicated rows from sports_management_agencies dataset
datasets["sports_management_agencies"].drop_duplicates(inplace=True)

# Check distributions and outliers
numerical_features = {}
for name, df in datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    numerical_features[name] = df[num_cols].describe()

# Display numerical summary for each dataset
for name, summary in numerical_features.items():
    tools.display_dataframe_to_user(name=f"Numerical Summary - {name}", dataframe=summary)


In [ ]:
# Set up visualization for numerical features
for name, df in datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(num_cols) > 0:
        plt.figure(figsize=(12, 6))
        df[num_cols].hist(bins=20, figsize=(12, 6))
        plt.suptitle(f"Distribution of Numerical Features - {name}", fontsize=14)
        plt.show()

# Boxplots for outlier detection
for name, df in datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(num_cols) > 0:
        plt.figure(figsize=(12, 6))
        df[num_cols].boxplot()
        plt.title(f"Boxplot of Numerical Features - {name}")
        plt.xticks(rotation=45)
        plt.show()


In [ ]:
# Function to remove outliers using IQR method
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Apply outlier removal for each dataset
cleaned_datasets = {}
for name, df in datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for col in num_cols:
        df = remove_outliers(df, col)
    cleaned_datasets[name] = df

# Display summary of data after outlier removal
summary_after_cleaning = {}
for name, df in cleaned_datasets.items():
    summary_after_cleaning[name] = {
        "Rows After Cleaning": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values (%)": (df.isnull().sum().sum() / df.size) * 100,
    }

# Display the cleaned dataset summary
cleaned_summary_df = pd.DataFrame(summary_after_cleaning).T
tools.display_dataframe_to_user(name="Summary After Outlier Removal", dataframe=cleaned_summary_df)


In [ ]:
# Visualize distributions after outlier removal
for name, df in cleaned_datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(num_cols) > 0:
        plt.figure(figsize=(12, 6))
        df[num_cols].hist(bins=20, figsize=(12, 6))
        plt.suptitle(f"Cleaned Distribution of Numerical Features - {name}", fontsize=14)
        plt.show()

# Boxplots after outlier removal
for name, df in cleaned_datasets.items():
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    if len(num_cols) > 0:
        plt.figure(figsize=(12, 6))
        df[num_cols].boxplot()
        plt.title(f"Boxplot After Outlier Removal - {name}")
        plt.xticks(rotation=45)
        plt.show()


In [ ]:
# Analyze categorical features
categorical_summary = {}
for name, df in cleaned_datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns
    categorical_summary[name] = {col: df[col].nunique() for col in cat_cols}

# Convert to DataFrame for better readability
categorical_summary_df = pd.DataFrame(categorical_summary).T
tools.display_dataframe_to_user(name="Categorical Feature Analysis", dataframe=categorical_summary_df)


In [ ]:
# Standardize categorical values (trimming spaces, converting to lowercase where needed)
for name, df in cleaned_datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        df[col] = df[col].str.strip().str.lower()

# Check for common inconsistencies in categorical data
categorical_issues = {}
for name, df in cleaned_datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns
    categorical_issues[name] = {col: df[col].value_counts().head(10) for col in cat_cols}

# Display common inconsistencies
for name, issues in categorical_issues.items():
    for col, counts in issues.items():
        tools.display_dataframe_to_user(name=f"Top Categories in {col} - {name}", dataframe=counts.to_frame())


In [ ]:
# Visualizing relationships between categorical and numerical variables
for name, df in cleaned_datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    
    if len(cat_cols) > 0 and len(num_cols) > 0:
        for cat_col in cat_cols[:2]:  # Limiting to first 2 categorical columns for readability
            for num_col in num_cols[:2]:  # Limiting to first 2 numerical columns for readability
                plt.figure(figsize=(12, 6))
                sns.boxplot(x=df[cat_col], y=df[num_col])
                plt.xticks(rotation=45)
                plt.title(f"Distribution of {num_col} by {cat_col} - {name}")
                plt.show()


In [ ]:
# Group similar categories in categorical features (if necessary)
def simplify_categories(df, column, threshold=5):
    """Groups categories that appear less than threshold times into 'Other'."""
    value_counts = df[column].value_counts()
    rare_categories = value_counts[value_counts < threshold].index
    df[column] = df[column].replace(rare_categories, "other")
    return df

# Apply to categorical columns with too many unique values
for name, df in cleaned_datasets.items():
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df[col].nunique() > 50:  # Arbitrary threshold for too many categories
            df = simplify_categories(df, col)
    cleaned_datasets[name] = df

# Display final cleaned dataset summaries
final_summary = {}
for name, df in cleaned_datasets.items():
    final_summary[name] = {
        "Final Rows": df.shape[0],
        "Final Columns": df.shape[1],
        "Unique Categories Reduced": sum(df.select_dtypes(include=['object']).nunique() < 50)
    }

# Display cleaned dataset summary
final_summary_df = pd.DataFrame(final_summary).T
tools.display_dataframe_to_user(name="Final Cleaned Dataset Summary", dataframe=final_summary_df)


In [ ]:
# Save the cleaned datasets for further use
for name, df in cleaned_datasets.items():
    file_path = f"/mnt/data/{name}_cleaned.csv"
    df.to_csv(file_path, index=False)

# Provide download links for cleaned datasets
cleaned_files = {name: f"/mnt/data/{name}_cleaned.csv" for name in cleaned_datasets.keys()}
cleaned_files
